# What is PySpark?

PySpark is a Python API for Apache Spark used to:

- Process large datasets
- Transform data
- Analyze data

In simple terms: **PySpark helps us work with big data efficiently**

---

# What is CRUD?

CRUD stands for:

| Operation | Meaning | PySpark Equivalent |
| --- | --- | --- |
| C | Create | Create DataFrame / Add rows |
| R | Read | Select / Filter |
| U | Update | withColumn |
| D | Delete | Filter (remove rows) |

---

# 1. CREATE → Add Data

### Purpose:

Create or add data into a DataFrame

In [0]:
# just recreating the data

columns = ['cust_id', 'cust_name', 'product', 'price']
data = [(1,'nilesh' ,'iphone', 200),
        (2,'pooja','dyson' , 1000),
        (3, 'raju','headphones' ,30)]

df=spark.createDataFrame(data,columns)
df.show()
# here lazy evaluation is taking place ---it wont display -- first it will decide - the sequence of execution of steps and then it will execute when .show() is executed


# we can add the rows -- not by update statement but by simply creating new row and then use union command

new_data_add = [(4,'shreyas','ipad',400)]
new_df = spark.createDataFrame(new_data_add,columns)
new_df.show()

updated=df.union(new_df)
updated.show()

In [0]:
data = [
    (1, 'emma wilson', 'UK', '2024-02-15'),
(2, 'john doe', 'USA', '2023-01-01'),
(3, 'sara jones', 'Canada', '2023-03-10'),
(4, 'mike brown', 'USA', '2023-02-05'),
(5, 'lisa lee', 'UK', '2023-04-20')
]


columns = ['customer_id','name', 'country','signup_date']


df = spark.createDataFrame(data,columns)
df.display()

In [0]:
#A PySpark DataFrame is immutable, so you don't directly insert a row into df. Instead, create another DataFrame for the new row and union it with the existing one.


new_data = [
    (6, 'alex smith', 'USA', '2023-05-01')
]

new_df = spark.createDataFrame(new_data,columns)
new_df.display()

df = df.union(new_df)
df.display()

## Key Points:

- PySpark DataFrames are **immutable**
- You don’t modify → you create a new DataFrame

## Practice:

- Add 2 new customers, from different countries

In [0]:
new_data2= [
    (7,'jane austen', 'uk', '2024-04-06'),
    (8,'jack london', 'canada', '2024-05-01')
]

new_df2= spark.createDataFrame(new_data2,columns)


df = df.union(new_df2)
df.display()

# 2. READ → View Data

### Purpose:

Retrieve data

In [0]:
updated.show()
updated.select('cust_id','cust_name').show()

#updated.filter("'price' < 3").show()
updated.printSchema()

updated.filter(" price >= 200").show()

In [0]:
df.show()
df.select('name','country').display()

In [0]:
df.filter("country = 'UK'").show()

## Key Points:

- `.show() and .display()` displays data
- `.select()` chooses columns
- `.filter()` applies conditions

## Practice:

- Show all customers
- Show only UK customers
- Show only names

In [0]:
df.select('name').show()
df.select('*').show()
df.filter("country = 'UK' ").show()

# 3. UPDATE → Modify Data

### Purpose:

Update existing data

In [0]:
from pyspark.sql.functions import when, lit
from pyspark.sql.functions import *

updated.show()
# first add a new column cust_type and assign 1 value to each row

updated= updated.withColumn('cust_type', lit(1))
updated.printSchema()
updated.show()

# now I will update the table column based on certain condition - okay --if price is > 200 cust_type will be 2

updated=updated.withColumn('cust_type', when(updated.price > 200 , lit(2)).otherwise(updated.cust_type))
updated.show()

updated=updated.withColumn('cust_type', when((updated.price > 200) & (updated.price < 500) , lit(3) ).otherwise(updated.cust_type))
updated.show()

In [0]:
from pyspark.sql.functions import when
from pyspark.sql.functions import lit
df = df.withColumn('country', when(df.customer_id == 1, 'india').otherwise(df.country))
df.display()


df = df.withColumn('source', lit('manual'))
df.display()

### Important:

- No direct UPDATE like SQL
- You recreate the column

### Key Points:

- Always assign back to `df`
- Logic is applied row-wise

### Practice:

- Update country for one customer
- Update multiple customers

In [0]:
df = df.withColumn('country',when (df.customer_id ==2,'Czech').otherwise(df.country))
df.show()

df = df.withColumn('source', when(df.source == 'manual', 'automatic').otherwise(df.source))
df.show()

# 4. DELETE → Remove Data

### Purpose:

Delete rows

In [0]:
df.filter(df.customer_id==1).show()

df1=df.filter(df.country != 'UK')
df1.show()

> Can remove almost all data accidentally

### Key Points:

- DELETE = filtering out rows
- No direct DELETE command

### Practice:

- Delete one customer
- Delete customers from UK

In [0]:
df.filter(df.customer_id==1).show()

df1=df.filter(df.country != 'UK')
df1.show()

# 5. RENAME COLUMNS

Use `withColumnRenamed()` when you want to change the name of an existing column. It returns a new DataFrame.


In [0]:
import pyspark.sql.functions as F

df=df.withColumn('signup_year',F.year('signup_date'))
df.display()

### Practice

- Rename `customer_name` back to `name`.


In [0]:
updated.show()

# now i need to rename the column from product to prod_name

updated = updated.withColumnRenamed('product','prod_name')
updated.show()

In [0]:
df = df.withColumnRenamed("country",'customer_country')
df.show()

# 6. DROP COLUMNS

Use `drop()` when a column is no longer required. You can drop one or multiple columns.


In [0]:
updated.show()
new_updated=updated.drop('cust_type')
new_updated.show()

In [0]:
df1=df.drop('signup_year')
df1.show()


### Practice

- Drop the `country` column.


In [0]:
df1=df.drop('customer_country')
df1.show()

# 7. SORT / ORDER BY

Use `orderBy()` to sort rows. By default, sorting is ascending. Use `desc()` for descending order.


In [0]:
import pyspark.sql.functions as F 
updated.show()
updated.orderBy(F.asc('price')).show()
updated.orderBy(F.desc('cust_type')).show()

In [0]:
df.orderBy(F.desc('customer_name')).display()

### Practice

- Sort customers by `customer_id` in ascending order.


In [0]:
df1=spark.read.table('rivadataplatform.landing.students')
df1.display()

In [0]:
df1.orderBy(F.desc('name')).display()

# 8. DISTINCT

Use `distinct()` to remove duplicate rows. You can also select a column first and find its unique values.


In [0]:
updated.show()
#updated.select('cust_type','price','cust_id').distinct().show()
updated.distinct().show()

In [0]:
#df.select('signup_year').distinct().show()
df1.select('city').distinct().show()

### Practice

- Show the unique customer countries.


In [0]:
df.select('customer_country').distinct().show()

# 9. NULL HANDLING

Use `dropna()` to remove rows containing NULL values and `fillna()` to replace NULL values.


In [0]:
updated.show()
new_data1 = [(5,'john','', 250,3)]

new_df1 = spark.createDataFrame(new_data1,columns)
updated1=updated.union(new_df1)
updated1.show()


updated1.fillna('unknown', subset = ['product']).show()

In [0]:
df1.show()

df=df.fillna('Unknown', subset = ['city'])
df.display()

### Practice

- Replace NULL values in `name` with `"Unknown"`.


# 10. GROUP BY AND AGGREGATIONS

Use `groupBy()` with functions such as `count()`, `sum()`, `avg()`, `min()` and `max()` to calculate summaries by group.


In [0]:
import pyspark.sql.functions as F
updated.show()
updated.groupBy('cust_type').agg(F.sum("price")).show()

In [0]:
df1.groupBy('city').agg(F.count('student_id')).show()
df1.groupBy('country').agg(F.count('student_id')).show()

### Practice

- Count how many customers are in each country.


In [0]:
df1.groupBy('country').agg(F.count('student_id')).show()

# 11. JOINS

`join()` combines two DataFrames using a matching key. Common types are `inner`, `left`, `right` and `full`.


In [0]:
df_attendance = spark.read.table('rivadataplatform.landing.attendance')
df_attendance.show()

df1.join(df_attendance, df1.student_id == df_attendance.student_id).show()

### Practice

- Perform an inner join between `df` and `orders` using `customer_id`.


# 12. UNION

Use `union()` or `unionByName()` to combine rows from two DataFrames. `unionByName()` matches columns by name and is generally safer when column order may differ.


### Practice

- Create a DataFrame with one new customer and combine it with `df` using `unionByName()`.


# 13. CAST DATA TYPES

Use `cast()` to change a column's data type. This is common when reading raw data.


In [0]:
updated.show()
updated.printSchema()
temp=updated.withColumn('cust_type',F.col('cust_type').cast('string'))
temp.show()
temp.printSchema()

In [0]:
df1 = df1.withColumn('student_id_int', F.col('student_id').cast('int'))
df1.withColumn('student_id_int',F.col('student_id_int').cast('int')).show()

### Practice

- Cast `customer_id` to `string` and inspect the schema.


# 14. STRING OPERATIONS

PySpark provides functions for common string transformations such as `upper()`, `lower()`, `trim()`, `length()` and `concat_ws()`.


In [0]:
df1.withColumn('length_cal', F.length('name')).show()
df1.withColumn('name_capital', F.upper('name')).show()


### Practice

- Create a new column containing the customer name in uppercase.


# 15. DATE OPERATIONS

Date functions are commonly used to extract year/month/day and calculate date differences.


In [0]:
df1.withColumn('year', F.year('created_at')).show()

### Practice

- Create a `signup_year` column from `signup_date`.


# 16. WINDOW FUNCTIONS

Window functions calculate values across related rows without collapsing them like `groupBy()`. They are useful for ranking, latest-record logic, `lag()` and `lead()`.


### Practice

- Create a row number for customers within each country ordered by `signup_date` descending.


# 17. TEMPORARY VIEW

A temporary view lets you query a DataFrame using SQL. It is session-scoped and is not stored as a permanent Unity Catalog object.


### Practice

- Create a temporary view called `uk_customers` containing only UK customers, then query it with SQL.


# 18. READ DATA

Spark can read common formats such as CSV, Parquet and Delta. In Databricks, Delta tables are especially common.


In [0]:
# Examples
# csv_df = spark.read.option("header", True).option("inferSchema", True).csv("/path/customers.csv")
# parquet_df = spark.read.parquet("/path/customers")
# delta_df = spark.read.table("catalog.schema.table_name")


In [0]:
updated.show()

df_csv=spark.read.option("header", True)\
    .option('inferSchema', True)\
    .csv('/Workspace/Users/nileshpatil508584@gmail.com/fmcg_childs_data/full_load_child/products/products.csv')

df_csv.display()

#df_parquet=spark.read.parquet('/Workspace/Users/nileshpatil508584@gmail.com/fmcg_childs_data/full_load_child/products/products.csv')
delta_df = spark.read.table("fmcg.bronze.customers")
delta_df.show()

### Practice

- Write the PySpark command you would use to read a Delta table named `catalog.schema.customers`.


# 19. WRITE DATA

Use `write` to persist a DataFrame. Common modes are `overwrite`, `append`, `ignore` and `error`.


In [0]:
# Example - write a DataFrame as a Delta table
# df.write.format("delta").mode("overwrite").saveAsTable("catalog.schema.customers")


df.write.format('delta')\
    .mode('append')\
        .saveAsTable("fmcg.silver.newly_added_customers")


### Practice

- Write `df` to a Delta table using append mode.


# 20. INSPECT A DATAFRAME

These commands are useful when developing and debugging PySpark transformations.


In [0]:
#updated.show()
#updated.printSchema()
print(updated.columns)
print(updated.dtypes)
print(updated.count())
updated.describe().display()

In [0]:
df.show()
df.printSchema()
print(df.columns)
print(df.dtypes)
print(df.count())
df.describe().show()


### Practice

- Display the schema, columns and row count for `df`.
